# Peaks Analysis From Confirmed Database

This notebook uses only confirmed database rows. Set `EXPERIMENT = None` for all confirmed rows, or set it to one available `canonical_id` / `experiment_id` in the setup cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks" and (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import importlib
import db_notebook_utils as dbu
importlib.reload(dbu)
from db_notebook_utils import (
    available_experiments,
    load_confirmed_data,
    load_single_experiment_context,
    significant_cluster_counts,
    load_cluster_mask,
)

# Set to None to analyze all confirmed database rows.
# Set to one canonical_id or experiment_id from available_experiments() for single-experiment analyses.
EXPERIMENT = None

experiments_df = available_experiments()
display(experiments_df)

data = load_confirmed_data(EXPERIMENT)
print(f"Loaded {len(data.cells)} confirmed cells, {len(data.peaks)} peak rows, {len(data.sta_results)} STA result rows, {len(data.clusters)} clusters.")
if data.cells.empty:
    raise ValueError("No confirmed cells are available for the selected database scope.")

def center_zero_ylim(ax, pad=0.08):
    ymin, ymax = ax.get_ylim()
    finite = [abs(v) for v in (ymin, ymax) if np.isfinite(v)]
    radius = max(finite) if finite else 1.0
    if radius == 0:
        radius = 1.0
    radius *= 1 + float(pad)
    ax.set_ylim(-radius, radius)



In [ ]:
# Confirmed peak table. These arrays come from confirmed_neuron_peaks, not from experiment folders.
peaks_df = data.peaks.copy()
peaks_df[["canonical_id", "cell_id", "n_all_peaks", "n_simple_peaks", "n_complex_peaks"]].head()


In [ ]:
summary = (
    peaks_df.groupby("canonical_id")
    .agg(
        n_cells=("cell_id", "nunique"),
        total_peaks=("n_all_peaks", "sum"),
        total_simple=("n_simple_peaks", "sum"),
        total_complex=("n_complex_peaks", "sum"),
        median_peaks_per_cell=("n_all_peaks", "median"),
    )
    .reset_index()
)
display(summary)


In [ ]:
fig, axes = plt.subplots(len(summary), 1, figsize=(12, max(3, 2.6 * len(summary))), squeeze=False, constrained_layout=True)
for ax, (canonical_id, exp_peaks) in zip(axes[:, 0], peaks_df.groupby("canonical_id", sort=False)):
    exp_peaks = exp_peaks.sort_values("n_all_peaks", ascending=False)
    max_peak = max((int(np.asarray(p).max()) for p in exp_peaks["all_peaks"] if np.asarray(p).size), default=0)
    for y, row in enumerate(exp_peaks.itertuples()):
        peaks = np.asarray(row.all_peaks, dtype=int)
        ax.vlines(peaks, y - 0.4, y + 0.4, color="black", linewidth=0.35)
    ax.set_title(f"{canonical_id}: confirmed spike rasters sorted by peak count")
    ax.set_ylabel("Confirmed cells")
    ax.set_yticks([])
    ax.set_xlim(0, max(1, max_peak))
    ax.set_xlabel("Frame")
plt.show()


In [ ]:
def spike_counts_for_group(peaks_df, bin_seconds=1.0):
    rows = []
    matrices = {}
    for (experiment_id, canonical_id), exp_peaks in peaks_df.groupby(["experiment_id", "canonical_id"], sort=False):
        all_peak_arrays = [np.asarray(v, dtype=np.int32) for v in exp_peaks["all_peaks"]]
        max_peak = max((int(p.max()) for p in all_peak_arrays if p.size), default=0)
        fps = 1.0
        # Correlations only need a stable bin width in frames; use 1 second when single-experiment fps is loaded elsewhere.
        bin_frames = max(1, int(round(bin_seconds * fps)))
        n_bins = max(1, int(np.ceil((max_peak + 1) / bin_frames)))
        cell_ids = exp_peaks["cell_id"].to_numpy(dtype=int)
        counts = np.zeros((len(cell_ids), n_bins), dtype=np.float32)
        for row_idx, peaks in enumerate(all_peak_arrays):
            bins = np.floor_divide(peaks, bin_frames)
            bins = bins[(bins >= 0) & (bins < n_bins)]
            if bins.size:
                counts[row_idx] = np.bincount(bins, minlength=n_bins).astype(np.float32)
        matrices[int(experiment_id)] = {"canonical_id": canonical_id, "cell_ids": cell_ids, "counts": counts}
    return matrices


def pairwise_spike_correlations(peaks_df, clusters_df, bin_frames=30):
    sig_counts = significant_cluster_counts(clusters_df)
    pair_rows = []
    for (experiment_id, canonical_id), exp_peaks in peaks_df.groupby(["experiment_id", "canonical_id"], sort=False):
        peak_arrays = {int(r.cell_id): np.asarray(r.all_peaks, dtype=np.int32) for r in exp_peaks.itertuples()}
        max_peak = max((int(p.max()) for p in peak_arrays.values() if p.size), default=0)
        n_bins = max(1, int(np.ceil((max_peak + 1) / max(1, bin_frames))))
        cell_ids = np.asarray(sorted(peak_arrays), dtype=int)
        counts = np.zeros((cell_ids.size, n_bins), dtype=np.float32)
        for i, cell_id in enumerate(cell_ids):
            bins = np.floor_divide(peak_arrays[int(cell_id)], bin_frames)
            bins = bins[(bins >= 0) & (bins < n_bins)]
            if bins.size:
                counts[i] = np.bincount(bins, minlength=n_bins).astype(np.float32)
        keep = counts.std(axis=1) > 0
        cell_ids = cell_ids[keep]
        counts = counts[keep]
        if counts.shape[0] < 2:
            continue
        corr = np.corrcoef(counts)
        for i, cell_i in enumerate(cell_ids):
            for j in range(i + 1, cell_ids.size):
                cell_j = int(cell_ids[j])
                pair_rows.append({
                    "experiment_id": int(experiment_id),
                    "canonical_id": canonical_id,
                    "cell_i": int(cell_i),
                    "cell_j": cell_j,
                    "correlation": float(corr[i, j]),
                    "n_spikes_i": int(peak_arrays[int(cell_i)].size),
                    "n_spikes_j": int(peak_arrays[cell_j].size),
                    "n_sig_clusters_i": int(sig_counts.get((int(experiment_id), int(cell_i)), 0)),
                    "n_sig_clusters_j": int(sig_counts.get((int(experiment_id), cell_j), 0)),
                })
    return pd.DataFrame(pair_rows)

pairwise_df = pairwise_spike_correlations(peaks_df, data.clusters, bin_frames=30)
print(f"Computed {len(pairwise_df)} within-experiment cell pairs.")
display(pairwise_df.head(20))


In [ ]:
if pairwise_df.empty:
    raise ValueError("No pairwise correlations could be computed from the selected confirmed rows.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].hist(pairwise_df["correlation"].dropna(), bins=30, color="#555555", edgecolor="white")
axes[0].axvline(pairwise_df["correlation"].median(), color="#D1495B", linestyle="--", label="median")
axes[0].set_title("Within-experiment spike-count correlations")
axes[0].set_xlabel("Pearson r")
axes[0].set_ylabel("Cell pairs")
axes[0].legend(frameon=False)

x = pairwise_df[["n_sig_clusters_i", "n_sig_clusters_j"]].min(axis=1)
y = pairwise_df["correlation"]
axes[1].scatter(x, y, alpha=0.65, s=28)
axes[1].set_title("Spike correlation vs shared significant-cluster availability")
axes[1].set_xlabel("Min significant clusters across pair")
axes[1].set_ylabel("Spike-count correlation")
axes[1].axhline(0, color="0.7", linewidth=1)
center_zero_ylim(axes[1])
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()


## Restored Detailed Pairwise Peak/Cluster Figures

These panels mirror the figure-focused analyses from the original notebook, using confirmed DB peak and STA-cluster rows.

In [ ]:
# Rebuild original-style pairwise columns from confirmed DB rows.
sig_cluster_sets = {
    (int(exp_id), int(cell_id)): set(g["cluster_id"].astype(int))
    for (exp_id, cell_id), g in data.clusters[data.clusters["is_significant"]].groupby(["experiment_id", "cell_id"])
}
peak_lookup = {
    (int(r.experiment_id), int(r.cell_id)): np.asarray(r.all_peaks, dtype=np.int32)
    for r in peaks_df.itertuples()
}

def count_peak_overlap(peaks_a, peaks_b, tolerance_frames=5):
    peaks_a = np.asarray(peaks_a, dtype=np.int32)
    peaks_b = np.asarray(peaks_b, dtype=np.int32)
    if peaks_a.size == 0 or peaks_b.size == 0:
        return 0
    pos = np.searchsorted(peaks_b, peaks_a)
    overlap = np.zeros(peaks_a.size, dtype=bool)
    valid = pos < peaks_b.size
    overlap[valid] |= np.abs(peaks_b[pos[valid]] - peaks_a[valid]) <= tolerance_frames
    valid_prev = pos > 0
    overlap[valid_prev] |= np.abs(peaks_b[pos[valid_prev] - 1] - peaks_a[valid_prev]) <= tolerance_frames
    return int(overlap.sum())

overlap_tolerance_frames = 5
for idx, row in pairwise_df.iterrows():
    key_i = (int(row.experiment_id), int(row.cell_i))
    key_j = (int(row.experiment_id), int(row.cell_j))
    peaks_i = peak_lookup[key_i]
    peaks_j = peak_lookup[key_j]
    clusters_i = sig_cluster_sets.get(key_i, set())
    clusters_j = sig_cluster_sets.get(key_j, set())
    shared = clusters_i & clusters_j
    union = clusters_i | clusters_j
    peak_overlap_count = min(count_peak_overlap(peaks_i, peaks_j, overlap_tolerance_frames), count_peak_overlap(peaks_j, peaks_i, overlap_tolerance_frames))
    pairwise_df.loc[idx, "peak_overlap_count"] = peak_overlap_count
    pairwise_df.loc[idx, "peak_overlap_fraction"] = peak_overlap_count / max(1, min(peaks_i.size, peaks_j.size))
    pairwise_df.loc[idx, "cluster_overlap_count"] = len(shared)
    pairwise_df.loc[idx, "cluster_overlap_fraction"] = len(shared) / max(1, min(len(clusters_i), len(clusters_j)))
    pairwise_df.loc[idx, "cluster_jaccard"] = len(shared) / len(union) if union else np.nan

summary_df = pd.DataFrame({
    "metric": [
        "neuron pairs",
        "median spike correlation",
        "median peak overlap fraction",
        "median cluster overlap fraction",
    ],
    "value": [
        int(len(pairwise_df)),
        float(pairwise_df["correlation"].median()),
        float(pairwise_df["peak_overlap_fraction"].median()),
        float(pairwise_df["cluster_overlap_fraction"].median()),
    ],
})
display(summary_df)
display(pairwise_df.sort_values("correlation", ascending=False).head(20))


In [ ]:
def experiment_spike_corr(exp_peaks, bin_frames=30):
    peak_arrays = {int(r.cell_id): np.asarray(r.all_peaks, dtype=np.int32) for r in exp_peaks.itertuples()}
    cell_ids = np.asarray(sorted(peak_arrays), dtype=int)
    max_peak = max((int(p.max()) for p in peak_arrays.values() if p.size), default=0)
    n_bins = max(1, int(np.ceil((max_peak + 1) / max(1, bin_frames))))
    counts = np.zeros((cell_ids.size, n_bins), dtype=np.float32)
    for i, cell_id in enumerate(cell_ids):
        bins = np.floor_divide(peak_arrays[int(cell_id)], bin_frames)
        bins = bins[(bins >= 0) & (bins < n_bins)]
        if bins.size:
            counts[i] = np.bincount(bins, minlength=n_bins).astype(np.float32)
    keep = counts.std(axis=1) > 0
    cell_ids = cell_ids[keep]
    counts = counts[keep]
    if cell_ids.size < 2:
        return cell_ids, None
    return cell_ids, np.corrcoef(counts)

for (experiment_id, canonical_id), exp_peaks in peaks_df.groupby(["experiment_id", "canonical_id"], sort=False):
    cell_ids, corr = experiment_spike_corr(exp_peaks, bin_frames=30)
    if corr is None:
        print(f"{canonical_id}: skipped heatmap, fewer than two variable cells.")
        continue
    n_sig = np.asarray([int(sig_cluster_sets.get((int(experiment_id), int(c)), set()).__len__()) for c in cell_ids])
    order = np.lexsort((cell_ids, -n_sig))
    fig = plt.figure(figsize=(13.5, 8), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 6], width_ratios=[6, 0.28])
    ax_bar = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[1, 0])
    cax = fig.add_subplot(gs[1, 1])
    rank = np.arange(cell_ids.size)
    ax_bar.bar(rank, n_sig[order], color="#3B6EA8", width=0.9)
    ax_bar.set_ylabel("Sig.\nclusters")
    ax_bar.set_title("Cells are ordered by number of significant astrocyte clusters")
    ax_bar.set_xlim(-0.5, cell_ids.size - 0.5)
    ax_bar.set_xticks([])
    im = ax_heat.imshow(corr[order][:, order], vmin=-1, vmax=1, cmap="RdBu_r", interpolation="nearest", aspect="equal")
    step = max(1, int(np.ceil(cell_ids.size / 18)))
    locs = np.arange(0, cell_ids.size, step)
    labels = cell_ids[order][locs]
    ax_heat.set_xticks(locs); ax_heat.set_xticklabels(labels, rotation=90)
    ax_heat.set_yticks(locs); ax_heat.set_yticklabels(labels)
    ax_heat.set_xlabel("Cell ID sorted by significant-cluster count")
    ax_heat.set_ylabel("Cell ID sorted by significant-cluster count")
    ax_heat.set_title("Spike-time correlation between neuron pairs")
    fig.colorbar(im, cax=cax, label="Pearson r\n30 frame spike-count bins")
    fig.suptitle(f"{canonical_id}: do neurons with more significant astrocyte clusters spike together?", y=1.02)
    plt.show()


In [ ]:
valid_overlap = pairwise_df.dropna(subset=["cluster_jaccard", "peak_overlap_fraction", "cluster_overlap_fraction"]).copy()
if valid_overlap.empty:
    print("No valid peak/cluster overlap rows to plot.")
else:
    peak_cluster_spearman = valid_overlap[["peak_overlap_fraction", "cluster_overlap_fraction"]].corr(method="spearman").iloc[0, 1]
    x = valid_overlap["peak_overlap_fraction"].to_numpy(dtype=float)
    y = valid_overlap["cluster_overlap_fraction"].to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    trend_x = np.linspace(0, max(0.001, np.nanmax(x)), 100)
    trend_y = None
    if finite.sum() >= 2 and np.nanstd(x[finite]) > 0:
        slope, intercept = np.polyfit(x[finite], y[finite], 1)
        trend_y = slope * trend_x + intercept
    bins = np.linspace(0, max(0.001, np.nanmax(x)), 7)
    bin_rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (valid_overlap["peak_overlap_fraction"] >= lo) & (valid_overlap["peak_overlap_fraction"] < hi)
        if in_bin.any():
            bin_rows.append({"x_mid": (lo + hi) / 2, "mean_cluster_overlap": valid_overlap.loc[in_bin, "cluster_overlap_fraction"].mean(), "sem_cluster_overlap": valid_overlap.loc[in_bin, "cluster_overlap_fraction"].sem(), "n_pairs": int(in_bin.sum())})
    binned = pd.DataFrame(bin_rows)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), constrained_layout=True)
    sc = axes[0].scatter(100 * valid_overlap["peak_overlap_fraction"], 100 * valid_overlap["cluster_overlap_fraction"], c=valid_overlap["cluster_overlap_count"], s=36 + 20 * valid_overlap["cluster_overlap_count"], cmap="magma_r", alpha=0.72, edgecolor="white", linewidth=0.45)
    if trend_y is not None:
        axes[0].plot(100 * trend_x, 100 * trend_y, color="#1B1B1B", linewidth=2, label="linear trend")
        axes[0].legend(frameon=False, loc="upper left")
    axes[0].set_title("Do shared spike times predict shared astrocyte clusters?")
    axes[0].set_xlabel(f"Spike-peak overlap (% of smaller peak set, +/-{overlap_tolerance_frames} frames)")
    axes[0].set_ylabel("Shared significant clusters (% of smaller cluster set)")
    axes[0].text(0.98, 0.05, f"Spearman r = {peak_cluster_spearman:.2f}\nEach point = one neuron pair", transform=axes[0].transAxes, ha="right", va="bottom", fontsize=10, bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.82"})
    fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label="Shared sig. clusters")

    if not binned.empty:
        axes[1].errorbar(100 * binned["x_mid"], 100 * binned["mean_cluster_overlap"], yerr=100 * binned["sem_cluster_overlap"].fillna(0), fmt="o-", color="#2A9D8F", ecolor="#8ABDB5", elinewidth=1.4, capsize=3, linewidth=2)
        for _, row in binned.iterrows():
            axes[1].annotate(f"n={int(row['n_pairs'])}", (100 * row["x_mid"], 100 * row["mean_cluster_overlap"]), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="0.3")
    axes[1].set_title("Binned summary of the same relationship")
    axes[1].set_xlabel("Spike-peak overlap bin center (%)")
    axes[1].set_ylabel("Mean shared-cluster overlap (%)")
    for ax in axes:
        ax.grid(color="0.92", linewidth=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Peak timing overlap vs significant-cluster overlap", y=1.04)
    plt.show()
    display(valid_overlap.sort_values(["cluster_overlap_fraction", "peak_overlap_fraction"], ascending=False).head(20))
